# Transportation Problem in Pure Python
## Five Exact Methods for Book Problem 2.7

This notebook presents five exact approaches for the transportation problem in pure Python:

1. Naive enumeration
2. Backtracking with pruning
3. Constraint-driven reduced search
4. Dynamic programming
5. Branch and Bound

We solve the base transportation model from book section `2.7`.

Each method reports:
- one optimal shipping plan
- the minimum transportation cost
- the execution time


In [1]:
from __future__ import annotations

from functools import lru_cache, wraps
from math import inf
from time import perf_counter


In [2]:
def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper


def total_cost_from_plan(plan, costs):
    total = 0
    for arc, shipped in plan.items():
        origin, destination = arc
        total += costs[(origin, destination)] * shipped
    return total


def choose_better(left, right):
    if left is None:
        return right
    if right is None:
        return left
    if right["total_cost"] < left["total_cost"]:
        return right
    if right["total_cost"] > left["total_cost"]:
        return left
    return right if tuple(sorted(right["shipments"].items())) < tuple(sorted(left["shipments"].items())) else left


## Problem: Transportation

Daar ships phones from:
- Cali plant
- Medellin plant
- Bogota warehouse

to six markets:
- Pasto
- Medellin
- Cartagena
- Cali
- Manizales
- Bogota

The book first asks to verify feasibility. Total supply is `1450` units and total demand is also `1450`, so the problem is balanced.

An exact structural reduction is possible here:
- Bogota is strictly cheapest for the Bogota market, and its warehouse capacity is `250`, so `250` units are sent from Bogota to Bogota in every optimum.
- After that, Cali is never needed for Medellin, Cartagena, or the remaining Bogota demand, so the exact search can focus on how Cali allocates its `600` units to Pasto, Cali, and Manizales.


In [3]:
ORIGINS = ["cali_plant", "medellin_plant", "bogota_warehouse"]
DESTINATIONS = ["pasto", "medellin", "cartagena", "cali", "manizales", "bogota"]

SUPPLY = {
    "cali_plant": 600,
    "medellin_plant": 600,
    "bogota_warehouse": 250,
}

DEMAND = {
    "pasto": 280,
    "medellin": 250,
    "cartagena": 240,
    "cali": 170,
    "manizales": 210,
    "bogota": 300,
}

COSTS = {
    ("cali_plant", "pasto"): 13150,
    ("cali_plant", "medellin"): 9650,
    ("cali_plant", "cartagena"): 19950,
    ("cali_plant", "cali"): 5000,
    ("cali_plant", "manizales"): 9650,
    ("cali_plant", "bogota"): 15950,
    ("medellin_plant", "pasto"): 19950,
    ("medellin_plant", "medellin"): 6500,
    ("medellin_plant", "cartagena"): 19950,
    ("medellin_plant", "cali"): 9650,
    ("medellin_plant", "manizales"): 11150,
    ("medellin_plant", "bogota"): 15950,
    ("bogota_warehouse", "pasto"): 17150,
    ("bogota_warehouse", "medellin"): 15950,
    ("bogota_warehouse", "cartagena"): 19950,
    ("bogota_warehouse", "cali"): 15950,
    ("bogota_warehouse", "manizales"): 15950,
    ("bogota_warehouse", "bogota"): 7000,
}

REDUCED_MARKETS = ["pasto", "cali", "manizales"]
REDUCED_BOUNDS = {
    "pasto": 280,
    "cali": 170,
    "manizales": 210,
}

CALI_DELTA = {
    "pasto": COSTS[("cali_plant", "pasto")] - COSTS[("medellin_plant", "pasto")],
    "cali": COSTS[("cali_plant", "cali")] - COSTS[("medellin_plant", "cali")],
    "manizales": COSTS[("cali_plant", "manizales")] - COSTS[("medellin_plant", "manizales")],
}

EXPECTED_SHIPMENTS = {
    ("cali_plant", "pasto"): 280,
    ("cali_plant", "cali"): 170,
    ("cali_plant", "manizales"): 150,
    ("medellin_plant", "medellin"): 250,
    ("medellin_plant", "cartagena"): 240,
    ("medellin_plant", "manizales"): 60,
    ("medellin_plant", "bogota"): 50,
    ("bogota_warehouse", "bogota"): 250,
}

EXPECTED_COST = 15609000

assert sum(SUPPLY.values()) == sum(DEMAND.values()) == 1450


In [4]:
def build_plan(cali_to_pasto, cali_to_cali, cali_to_manizales):
    if cali_to_pasto < 0 or cali_to_cali < 0 or cali_to_manizales < 0:
        return None
    if cali_to_pasto > REDUCED_BOUNDS["pasto"] or cali_to_cali > REDUCED_BOUNDS["cali"] or cali_to_manizales > REDUCED_BOUNDS["manizales"]:
        return None
    if cali_to_pasto + cali_to_cali + cali_to_manizales != 600:
        return None

    plan = {
        ("cali_plant", "pasto"): cali_to_pasto,
        ("cali_plant", "cali"): cali_to_cali,
        ("cali_plant", "manizales"): cali_to_manizales,
        ("medellin_plant", "pasto"): 280 - cali_to_pasto,
        ("medellin_plant", "medellin"): 250,
        ("medellin_plant", "cartagena"): 240,
        ("medellin_plant", "cali"): 170 - cali_to_cali,
        ("medellin_plant", "manizales"): 210 - cali_to_manizales,
        ("medellin_plant", "bogota"): 50,
        ("bogota_warehouse", "bogota"): 250,
    }

    if any(value < 0 for value in plan.values()):
        return None

    filtered = {arc: shipped for arc, shipped in plan.items() if shipped > 0}
    return {
        "shipments": filtered,
        "total_cost": total_cost_from_plan(filtered, COSTS),
    }


def savings(values):
    return sum(-CALI_DELTA[market] * values[market] for market in REDUCED_MARKETS)


BASELINE_PLAN = {
    ("medellin_plant", "pasto"): 280,
    ("medellin_plant", "medellin"): 250,
    ("medellin_plant", "cartagena"): 240,
    ("medellin_plant", "cali"): 170,
    ("medellin_plant", "manizales"): 210,
    ("medellin_plant", "bogota"): 50,
    ("bogota_warehouse", "bogota"): 250,
}

BASELINE_COST = total_cost_from_plan(BASELINE_PLAN, COSTS)


In [5]:
@timed
def solve_transport_naive():
    best = None
    for cali_to_pasto in range(REDUCED_BOUNDS["pasto"] + 1):
        for cali_to_cali in range(REDUCED_BOUNDS["cali"] + 1):
            cali_to_manizales = 600 - cali_to_pasto - cali_to_cali
            candidate = build_plan(cali_to_pasto, cali_to_cali, cali_to_manizales)
            best = choose_better(best, candidate)
    return best


In [6]:
@timed
def solve_transport_backtracking():
    best = None

    ordered_markets = ["pasto", "cali", "manizales"]

    def optimistic_cost(prefix_values, remaining_supply, index):
        current_delta_cost = sum(CALI_DELTA[market] * prefix_values.get(market, 0) for market in prefix_values)
        future_delta_cost = 0
        supply_left = remaining_supply
        for future_market in ordered_markets[index:]:
            take = min(REDUCED_BOUNDS[future_market], supply_left)
            future_delta_cost += CALI_DELTA[future_market] * take
            supply_left -= take
        return BASELINE_COST + current_delta_cost + future_delta_cost

    def backtrack(index, remaining_supply, chosen):
        nonlocal best
        if index == len(ordered_markets) - 1:
            chosen[ordered_markets[index]] = remaining_supply
            candidate = build_plan(
                chosen["pasto"],
                chosen["cali"],
                chosen["manizales"],
            )
            best = choose_better(best, candidate)
            return

        market = ordered_markets[index]
        lower = max(0, remaining_supply - sum(REDUCED_BOUNDS[m] for m in ordered_markets[index + 1 :]))
        upper = min(REDUCED_BOUNDS[market], remaining_supply)

        for value in range(upper, lower - 1, -1):
            chosen[market] = value
            lower_bound_cost = optimistic_cost(chosen, remaining_supply - value, index + 1)
            if best is not None and lower_bound_cost >= best["total_cost"]:
                continue
            backtrack(index + 1, remaining_supply - value, chosen)
        chosen.pop(market, None)

    backtrack(0, 600, {})
    return best


In [7]:
@timed
def solve_transport_reduced_search():
    remaining_supply = 600
    chosen = {}
    for market in sorted(REDUCED_MARKETS, key=lambda name: CALI_DELTA[name]):
        take = min(REDUCED_BOUNDS[market], remaining_supply)
        chosen[market] = take
        remaining_supply -= take
    return build_plan(chosen["pasto"], chosen["cali"], chosen["manizales"])


In [8]:
@timed
def solve_transport_dp():
    markets = tuple(REDUCED_MARKETS)

    @lru_cache(maxsize=None)
    def dp(index, remaining_supply):
        if index == len(markets):
            if remaining_supply == 0:
                return 0, {}
            return -inf, None

        market = markets[index]
        best_savings = -inf
        best_choice = None
        lower = max(0, remaining_supply - sum(REDUCED_BOUNDS[m] for m in markets[index + 1 :]))
        upper = min(REDUCED_BOUNDS[market], remaining_supply)

        for value in range(lower, upper + 1):
            suffix_savings, suffix_choice = dp(index + 1, remaining_supply - value)
            if suffix_choice is None:
                continue
            current_savings = -CALI_DELTA[market] * value + suffix_savings
            if current_savings > best_savings:
                best_savings = current_savings
                best_choice = {market: value, **suffix_choice}
        return best_savings, best_choice

    _, chosen = dp(0, 600)
    return build_plan(chosen["pasto"], chosen["cali"], chosen["manizales"])


In [9]:
@timed
def solve_transport_branch_and_bound():
    best = None
    ordered_markets = tuple(sorted(REDUCED_MARKETS, key=lambda name: CALI_DELTA[name]))
    stack = [(0, 600, {})]

    while stack:
        index, remaining_supply, chosen = stack.pop()

        if index == len(ordered_markets) - 1:
            trial = dict(chosen)
            trial[ordered_markets[index]] = remaining_supply
            candidate = build_plan(trial["pasto"], trial["cali"], trial["manizales"])
            best = choose_better(best, candidate)
            continue

        market = ordered_markets[index]
        lower = max(0, remaining_supply - sum(REDUCED_BOUNDS[m] for m in ordered_markets[index + 1 :]))
        upper = min(REDUCED_BOUNDS[market], remaining_supply)

        for value in range(upper, lower - 1, -1):
            trial = dict(chosen)
            trial[market] = value
            optimistic_savings = sum(-CALI_DELTA[m] * trial.get(m, 0) for m in trial)
            rem = remaining_supply - value
            for future_market in ordered_markets[index + 1 :]:
                take = min(REDUCED_BOUNDS[future_market], rem)
                optimistic_savings += -CALI_DELTA[future_market] * take
                rem -= take

            lower_bound_cost = BASELINE_COST - optimistic_savings
            if best is not None and lower_bound_cost >= best["total_cost"]:
                continue
            stack.append((index + 1, remaining_supply - value, trial))

    return best


In [10]:
naive_result, naive_time = solve_transport_naive()
backtracking_result, backtracking_time = solve_transport_backtracking()
reduced_result, reduced_time = solve_transport_reduced_search()
dp_result, dp_time = solve_transport_dp()
bb_result, bb_time = solve_transport_branch_and_bound()

print("=== TRANSPORTATION RESULTS ===")
print(f"Balanced problem check       -> supply = {sum(SUPPLY.values())}, demand = {sum(DEMAND.values())}")
print(f"Naive enumeration            -> {naive_result}, time = {naive_time:.8f} seconds")
print(f"Backtracking with pruning    -> {backtracking_result}, time = {backtracking_time:.8f} seconds")
print(f"Constraint-driven reduced search -> {reduced_result}, time = {reduced_time:.8f} seconds")
print(f"Dynamic programming          -> {dp_result}, time = {dp_time:.8f} seconds")
print(f"Branch and Bound             -> {bb_result}, time = {bb_time:.8f} seconds")

assert naive_result["shipments"] == EXPECTED_SHIPMENTS
assert naive_result["total_cost"] == EXPECTED_COST
assert backtracking_result == naive_result
assert reduced_result == naive_result
assert dp_result == naive_result
assert bb_result == naive_result
print("All five exact methods agree with the textbook transportation plan.")


=== TRANSPORTATION RESULTS ===
Balanced problem check       -> supply = 1450, demand = 1450
Naive enumeration            -> {'shipments': {('cali_plant', 'pasto'): 280, ('cali_plant', 'cali'): 170, ('cali_plant', 'manizales'): 150, ('medellin_plant', 'medellin'): 250, ('medellin_plant', 'cartagena'): 240, ('medellin_plant', 'manizales'): 60, ('medellin_plant', 'bogota'): 50, ('bogota_warehouse', 'bogota'): 250}, 'total_cost': 15609000}, time = 0.01138500 seconds
Backtracking with pruning    -> {'shipments': {('cali_plant', 'pasto'): 280, ('cali_plant', 'cali'): 170, ('cali_plant', 'manizales'): 150, ('medellin_plant', 'medellin'): 250, ('medellin_plant', 'cartagena'): 240, ('medellin_plant', 'manizales'): 60, ('medellin_plant', 'bogota'): 50, ('bogota_warehouse', 'bogota'): 250}, 'total_cost': 15609000}, time = 0.00529738 seconds
Constraint-driven reduced search -> {'shipments': {('cali_plant', 'pasto'): 280, ('cali_plant', 'cali'): 170, ('cali_plant', 'manizales'): 150, ('medellin_pla

## Variants in the Book

Section `2.7` does not add a student variation after the base transportation model, so this notebook closes with the balanced base case only.
